# 单谱基线校正对照

这个 notebook 用来查看宇宙射线清理之后，进入基线估计和基线扣除阶段的效果

展示内容：

1. 原始谱和宇宙射线清理后的光谱
2. 宇宙射线清理后光谱与估计基线
3. 基线扣除后的光谱
4. 可选对比 `asls / arpls / airpls` 三种基线方法

默认从 `DATASET_NAME` 对应数据集的 `init` 目录里随机抽取一条原始谱。如果要固定单条样本，填 `SAMPLE_PATH`；如果要固定某个文件夹内随机抽样，填 `SAMPLE_FOLDER`


In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


def find_project_root(start=None):
    """从当前目录向上查找项目根目录，兼容从 notebooks/ 里启动"""
    current = Path(start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "raman").is_dir() and (path / "dataset").is_dir():
            return path
    return current


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "raman" or name.startswith("raman."):
        del sys.modules[name]

from raman.data.build import COMMON_BAD_BANDS, DEFAULT_PIPELINE_CONFIG
from raman.data.offline import estimate_baseline, remove_cosmic_rays
from raman.data.profiles import get_dataset_dir, get_profile
from raman.data.spectrum import build_valid_mask, read_arc_data


sns.set_theme(style="whitegrid", context="talk", font="Microsoft YaHei")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]

cfg = DEFAULT_PIPELINE_CONFIG

DATASET_NAME = "细菌"
SAMPLE_PATH = "dataset/细菌/init/Proteus/PVU01/cell7_Area01_000_shift.arc_data"  # 填文件路径时固定单条样本；None 时走随机抽样
SAMPLE_FOLDER = None  # 填相对文件夹时只在该文件夹内随机抽样；None 时用 init
SAMPLE_SEED = None  # 设为整数可复现随机抽样

BASELINE_METHOD = None  # None 表示使用 raman.data.build.DEFAULT_PIPELINE_CONFIG.baseline_method
COMPARE_BASELINES = True
DISPLAY_MIN = None  # baseline 阶段还没有裁切，默认展示完整波数轴
DISPLAY_MAX = None
BAD_BANDS = COMMON_BAD_BANDS

profile = get_profile(DATASET_NAME)
dataset_dir = PROJECT_ROOT / "dataset" / DATASET_NAME
if not dataset_dir.exists():
    dataset_dir = get_dataset_dir(profile, PROJECT_ROOT)
method = (BASELINE_METHOD or cfg.baseline_method).lower()


def resolve_sample_path():
    """根据显式路径、指定文件夹或随机策略选出一条原始谱"""
    if SAMPLE_PATH:
        path = Path(SAMPLE_PATH)
        return path if path.is_absolute() else PROJECT_ROOT / path

    root = Path(SAMPLE_FOLDER) if SAMPLE_FOLDER else Path(profile.root_init)
    if not root.is_absolute():
        root = dataset_dir / root

    matches = sorted(root.rglob("*.arc_data"))
    if not matches:
        raise FileNotFoundError("未找到可展示的 .arc_data 文件，请先准备 init")

    rng = random.Random(SAMPLE_SEED)
    return rng.choice(matches)


def add_bad_band_spans(ax, bad_bands):
    """用灰色区域标记坏波段"""
    for i, (band_min, band_max) in enumerate(bad_bands):
        ax.axvspan(
            band_min,
            band_max,
            color="gray",
            alpha=0.18,
            label="bad band" if i == 0 else None,
        )


def plot_without_bad_bands(ax, wn, y, bad_bands, **kwargs):
    """坏段内不画线，坏段两侧也不连线，并保持同一曲线颜色一致"""
    wn = np.asarray(wn, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    if "color" not in kwargs:
        kwargs["color"] = ax._get_lines.get_next_color()
    keep_mask = build_valid_mask(wn, bad_bands)
    if keep_mask is None:
        keep_mask = np.ones_like(wn, dtype=bool)

    start = None
    for idx, keep in enumerate(keep_mask):
        if keep and start is None:
            start = idx
        elif (not keep) and start is not None:
            ax.plot(wn[start:idx], y[start:idx], **kwargs)
            kwargs.pop("label", None)
            start = None
    if start is not None:
        ax.plot(wn[start:], y[start:], **kwargs)


def style_axis(ax, title, ylabel="Intensity"):
    """统一单谱图的坐标轴样式"""
    add_bad_band_spans(ax, BAD_BANDS)
    if DISPLAY_MIN is not None and DISPLAY_MAX is not None:
        ax.set_xlim(DISPLAY_MIN, DISPLAY_MAX)
    ax.set_title(title)
    ax.set_xlabel("Wavenumber (cm$^{-1}$)")
    ax.set_ylabel(ylabel)
    ax.legend(loc="best")


sample_path = resolve_sample_path()
wn, sp = read_arc_data(sample_path)
if wn.size == 0 or sp.size == 0:
    raise ValueError(f"读取失败：{sample_path}")

cosmic_valid_mask = np.ones_like(wn, dtype=bool)

sp_cosmic, cosmic_stats = remove_cosmic_rays(
    sp,
    window_cm=cfg.cosmic_ray_narrow_window_cm,
    threshold=cfg.cosmic_ray_threshold,
    max_iter=cfg.cosmic_ray_max_iter,
    valid_mask=cosmic_valid_mask,
    wn=wn,
    peak_prominence_z=cfg.cosmic_ray_peak_prominence_z,
    peak_width_max_cm=cfg.cosmic_ray_peak_width_max_cm,
    peak_ratio_z_per_cm=cfg.cosmic_ray_peak_ratio_z_per_cm,
    peak_pad_cm=cfg.cosmic_ray_peak_pad_cm,
    peak_rel_height=cfg.cosmic_ray_peak_rel_height,
    residual_threshold_z=cfg.cosmic_ray_residual_threshold_z,
    residual_max_points_fraction=cfg.cosmic_ray_residual_max_points_fraction,
)

baseline_fit_mask = (wn >= cfg.baseline_fit_min) & (wn <= cfg.baseline_fit_max)
wn_fit = wn[baseline_fit_mask]
sp_cosmic_fit = sp_cosmic[baseline_fit_mask]
valid_fit_mask = build_valid_mask(wn_fit, BAD_BANDS)
if valid_fit_mask is None:
    valid_fit_mask = np.ones_like(wn_fit, dtype=bool)
if wn_fit.size < 10:
    raise ValueError("baseline 拟合窗口内有效点过少")

baseline = estimate_baseline(
    sp_cosmic_fit,
    method=method,
    lam=cfg.baseline_lam,
    p=cfg.baseline_asls_p,
    niter=cfg.baseline_max_iter,
    valid_mask=valid_fit_mask,
)
sp_corrected = sp_cosmic_fit - baseline

print(f"project_root = {PROJECT_ROOT}")
print(f"dataset_dir = {dataset_dir}")
print(f"sample_path = {sample_path}")
print(f"bad_bands = {BAD_BANDS}")
print(f"cosmic narrow replaced = {cosmic_stats.narrow}")
print(f"cosmic peak-morph replaced = {cosmic_stats.peak}")
print(f"cosmic residual replaced = {cosmic_stats.residual}")
print(f"cosmic total replaced = {cosmic_stats.total}")
print(f"baseline method = {method}")
print(f"baseline fit range = {cfg.baseline_fit_min}-{cfg.baseline_fit_max} cm^-1")
print(f"baseline_lam = {cfg.baseline_lam}")
print(f"baseline_asls_p = {cfg.baseline_asls_p}")
print(f"baseline_max_iter = {cfg.baseline_max_iter}")


In [ ]:
# 1. 查看宇宙射线清理前后，确认进入基线阶段的光谱形态
fig, ax = plt.subplots(figsize=(12, 6))
plot_without_bad_bands(ax, wn, sp, BAD_BANDS, label="raw", alpha=0.45, linewidth=1.0)
plot_without_bad_bands(ax, wn, sp_cosmic, BAD_BANDS, label="after cosmic removal", alpha=0.9, linewidth=1.2)
style_axis(ax, "原始谱 / 宇宙射线清理后")
plt.show()


In [ ]:
# 2. 查看 400-2000 cm^-1 拟合窗口内的基线是否贴合低频背景，而不是吃掉 Raman 峰
fig, ax = plt.subplots(figsize=(12, 6))
plot_without_bad_bands(ax, wn_fit, sp_cosmic_fit, BAD_BANDS, label="after cosmic removal", alpha=0.85, linewidth=1.1)
plot_without_bad_bands(ax, wn_fit, baseline, BAD_BANDS, label=f"{method} baseline", alpha=0.95, linewidth=2.0)
style_axis(ax, f"{method} 基线估计")
plt.show()


In [ ]:
# 3. 查看基线扣除后的结果；正式流程随后再裁到 600-1800 cm^-1、删除坏段并插值
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
plot_without_bad_bands(axes[0], wn_fit, sp_cosmic_fit, BAD_BANDS, label="after cosmic removal", color="C0", linewidth=1.1)
plot_without_bad_bands(axes[0], wn_fit, baseline, BAD_BANDS, label=f"{method} baseline", color="C1", linewidth=2.0)
style_axis(axes[0], "基线扣除前", ylabel="Intensity")

plot_without_bad_bands(axes[1], wn_fit, sp_corrected, BAD_BANDS, label="baseline corrected", color="C2", linewidth=1.1)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.6)
style_axis(axes[1], "基线扣除后", ylabel="Corrected intensity")
fig.tight_layout()
plt.show()


In [ ]:
# 4. 可选：横向比较 asls / arpls / airpls 三种基线估计
if COMPARE_BASELINES:
    methods = ["asls", "airpls"]
    baselines = {}
    corrected = {}
    for item in methods:
        baselines[item] = estimate_baseline(
            sp_cosmic_fit,
            method=item,
            lam=cfg.baseline_lam,
            p=cfg.baseline_asls_p,
            niter=cfg.baseline_max_iter,
            valid_mask=valid_fit_mask,
        )
        corrected[item] = sp_cosmic_fit - baselines[item]

    fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    plot_without_bad_bands(axes[0], wn_fit, sp_cosmic_fit, BAD_BANDS, label="after cosmic removal", color="0.65", linewidth=1.0)
    for item in methods:
        plot_without_bad_bands(axes[0], wn_fit, baselines[item], BAD_BANDS, label=f"{item} baseline", linewidth=1.8)
    style_axis(axes[0], "不同 baseline 方法的基线形态", ylabel="Intensity")

    for item in methods:
        plot_without_bad_bands(axes[1], wn_fit, corrected[item], BAD_BANDS, label=f"{item} corrected", linewidth=1.0)
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.6)
    style_axis(axes[1], "不同 baseline 方法的扣除结果", ylabel="Corrected intensity")
    fig.tight_layout()
    plt.show()
else:
    print("COMPARE_BASELINES=False，跳过方法对比")
